# Jogadores — `padel_analytics` (só o PlayerTracker)

Corre **apenas** a deteção de jogadores do João Silva (`yolov8m` + `polygon_zone` + **ByteTrack**)
sobre o `Parada4.mp4`, e descarrega o `players_detections.json` com as **boxes em píxeis**.

**Não treina nada. Não improvisa detetor nenhum.** É ligar o que já existe.

Porquê só este tracker (e não o `main.py` inteiro): o `main.py` corre 4 modelos e obriga a
clicar 12 keypoints do campo (só precisos para a homografia → metros). Nós queremos **píxeis**.
Aqui só precisas de **4 cliques** (os cantos), e corre em ~1/4 do tempo.

Antes de começar: `Runtime → Change runtime type → T4 GPU`.


## 1. GPU + clonar o repo do João

In [ ]:
!nvidia-smi -L
%cd /content
![ -d padel_analytics ] || git clone https://github.com/Joao-M-Silva/padel_analytics.git
%cd /content/padel_analytics
!pip install -q ultralytics supervision opencv-python-headless gdown
print('OK')

## 2. Pesos (só o dos jogadores: `yolov8m.pt`)
Vem da Drive do João (a mesma pasta do outro notebook).

In [ ]:
import gdown, os, glob, shutil
FOLDER = 'https://drive.google.com/drive/folders/1joO7w1Am7B418SIqGBq90YipQl81FMzh'
if not os.path.exists('weights/players_detection/yolov8m.pt'):
    gdown.download_folder(FOLDER, output='_weights_dl', quiet=True, use_cookies=False)
    found = {os.path.basename(p): p for p in glob.glob('_weights_dl/**/*.pt', recursive=True)}
    os.makedirs('weights/players_detection', exist_ok=True)
    assert 'yolov8m.pt' in found, f'Falta o yolov8m.pt. Encontrei: {list(found)}'
    shutil.copy(found['yolov8m.pt'], 'weights/players_detection/yolov8m.pt')
print('Pesos OK:', os.path.exists('weights/players_detection/yolov8m.pt'))

## 3. Vídeo — faz upload do `Parada4.mp4`
(É o vídeo do ground-truth: 12 rallies / 117s.)

In [ ]:
from google.colab import files
import os
if not os.path.exists('/content/Parada4.mp4'):
    up = files.upload()          # escolhe o Parada4.mp4
    src = list(up.keys())[0]
    os.rename(src, '/content/Parada4.mp4')
VIDEO_PATH = '/content/Parada4.mp4'

import cv2
cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS)
N   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'{VIDEO_PATH}: {W}x{H}, {FPS:.2f} fps, {N} frames ({N/FPS:.0f}s)')

## 4. O campo — vem da NOSSA calibração (não se clica nada)

Faz upload do **`calibracao_campo.json`** (o que desenhaste no `calibrar_campo.html`).

O polígono que filtra o público sai **das linhas que já desenhaste**: o ponto onde cada linha
encontra o vidro **é** o canto do campo. Clicar cantos outra vez seria repetir, pior, o que já está feito.

Vantagem sobre os 4 cantos do `padel_analytics`: o campo real é **curvo** (a lente), e um
quadrilátero de 4 vértices ou corta jogadores nas pontas ou deixa entrar bancada.
Aqui o polígono acompanha a curva, com **40 px de margem** — a diretriz manda **nunca perder um jogador**.


In [ ]:
from google.colab import files
import json, numpy as np, os

if not os.path.exists('/content/calibracao_campo.json'):
    print('Escolhe o calibracao_campo.json (da pasta padelpro-vision/)')
    up = files.upload()
    os.rename(list(up.keys())[0], '/content/calibracao_campo.json')

cal = json.load(open('/content/calibracao_campo.json'))
P = cal['pontos']

# As LATERAIS saem dos EXTREMOS das linhas horizontais: onde a linha encontra o vidro = o canto.
esq, dir_ = [], []
for k in ('fundo_longe', 'servico_longe', 'rede_base', 'servico_perto'):
    q = sorted(P[k], key=lambda t: t[0])
    esq.append((q[0][1], q[0][0]))
    dir_.append((q[-1][1], q[-1][0]))
E, D = np.array(esq), np.array(dir_)
c_esq = np.polyfit(E[:, 0], E[:, 1], 2)
c_dir = np.polyfit(D[:, 0], D[:, 1], 2)

MARGEM = 40                      # generosa: os jogadores vao aos espacos laterais
y_fundo = float(np.polyval(cal['fundo_longe_coef'], 480))
W, H = 960, 540

ys = list(range(int(y_fundo) - 6, H - 1, 10)) + [H - 1]
poly = [[np.polyval(c_esq, y) - MARGEM, y] for y in ys]
poly += [[np.polyval(c_dir, y) + MARGEM, y] for y in reversed(ys)]
CAMPO = np.array([[int(np.clip(x, 0, W - 1)), int(np.clip(y, 0, H - 1))] for x, y in poly], dtype=np.int32)

print(f'Poligono do campo: {len(CAMPO)} vertices, margem {MARGEM}px')

# ver
import cv2
from google.colab.patches import cv2_imshow
cap = cv2.VideoCapture(VIDEO_PATH); cap.set(cv2.CAP_PROP_POS_FRAMES, 1400)
ok, img = cap.read(); cap.release()
if ok:
    ov = img.copy(); cv2.fillPoly(ov, [CAMPO], (0, 255, 255))
    img = cv2.addWeighted(ov, 0.25, img, 0.75, 0)
    cv2.polylines(img, [CAMPO], True, (0, 255, 255), 2)
    cv2_imshow(img)
print('\nCONFIRMA: o amarelo cobre o campo todo e deixa a bancada de fora?')

### 4b. Fallback — clicar 4 cantos à mão
Só se não tiveres a calibração. Preenche e corre.

In [ ]:
# import numpy as np
# CANTOS = [[950,530],[10,530],[330,45],[700,45]]   # perto-DIR, perto-ESQ, longe-ESQ, longe-DIR
# CAMPO = np.array(CANTOS, dtype=np.int32); print(CAMPO)

## 5. Correr **só** o PlayerTracker do João

Importa a classe `PlayerTracker` do repo dele — **sem a reescrever**.
`yolov8m` + `polygon_zone` (os teus 4 cantos) + `ByteTrack` (IDs estáveis).

In [ ]:
%cd /content/padel_analytics
import numpy as np, supervision as sv
from tqdm import tqdm
from trackers.players_tracker.players_tracker import PlayerTracker

video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
polygon_zone = sv.PolygonZone(CAMPO, frame_resolution_wh=video_info.resolution_wh)

tracker = PlayerTracker(
    'weights/players_detection/yolov8m.pt',
    polygon_zone,
    batch_size=8,
    save_path=None, load_path=None,
)
tracker.video_info_post_init(video_info)
print('device:', tracker.DEVICE, '| frames:', video_info.total_frames)
print('SEM max_det -> deteta toda a gente, o poligono e que descarta. (O bug de ontem foi cortar aos 4.)')

BATCH = 8
todos, lote = [], []
for frame in tqdm(sv.get_video_frames_generator(VIDEO_PATH), total=video_info.total_frames):
    lote.append(frame)
    if len(lote) == BATCH:
        todos.extend(tracker.predict_sample(lote)); lote = []
if lote:
    todos.extend(tracker.predict_sample(lote))

print('frames processados:', len(todos))

## 6. Gravar o JSON (boxes em **píxeis** + tracker_id) e ver a cobertura

In [ ]:
import json
serial = [p.serialize() for p in todos]
OUT = '/content/players_detections_parada4.json'
with open(OUT, 'w') as f: json.dump(serial, f)

n = len(serial) or 1
n4 = sum(1 for fr in serial if len(fr) >= 4)
n3 = sum(1 for fr in serial if len(fr) >= 3)
n1 = sum(1 for fr in serial if len(fr) >= 1)
print(f'frames            : {len(serial)}')
print(f'com os 4 jogadores: {100*n4/n:.1f}%   <-- o numero que interessa (yolov8n improvisado: 53%)')
print(f'com >=3           : {100*n3/n:.1f}%')
print(f'com >=1           : {100*n1/n:.1f}%')
print(f'media por frame   : {sum(len(fr) for fr in serial)/n:.2f}')
print('\nexemplo (frame 200):', serial[200][:1])

## 6b. ⭐ VER com os olhos — vídeo de um rally com as caixas

Antes de acreditar em qualquer número: **olha.** Isto corta um rally do ground-truth e desenha
as caixas por cima, com o **ID do ByteTrack** e o **ponto dos pés**.

O que confirmar:
1. **São 4 caixas** — e são os 4 jogadores, não público.
2. **Os IDs não trocam** quando os jogadores se cruzam ou se tapam.
3. **Os pés** (o ponto) estão no chão, dentro do campo — é o ponto que todas as regras usam.

In [ ]:
# escolhe um rally do ground-truth (t_ini, t_fim em segundos)
T_INI, T_FIM = 46.8, 67.5     # rally #2, o mais longo (20.7s)
# outros: (38.0,41.5) (77.6,85.5) (95.9,111.1) (157.9,169.4)

import cv2, numpy as np
from tqdm import tqdm

f0, f1 = int(T_INI*video_info.fps), int(T_FIM*video_info.fps)
CORES = [(0,255,0),(0,180,255),(255,120,0),(255,0,255),(0,0,255),(255,255,0)]

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
w, h = video_info.resolution_wh
out = cv2.VideoWriter('/content/rally_boxes.mp4', cv2.VideoWriter_fourcc(*'mp4v'),
                      video_info.fps, (w, h))

n_por_frame = []
for f in tqdm(range(f0, f1)):
    ok, img = cap.read()
    if not ok: break
    players = todos[f] if f < len(todos) else []
    n_por_frame.append(len(players))
    for p in players:
        x1, y1, x2, y2 = [int(v) for v in p.xyxy]
        cor = CORES[(p.id or 0) % len(CORES)]
        cv2.rectangle(img, (x1,y1), (x2,y2), cor, 2)
        cv2.circle(img, p.feet, 5, cor, -1)              # PES = o que as regras usam
        cv2.putText(img, f'#{p.id}', (x1, y1-6), 0, 0.6, cor, 2)
    cv2.putText(img, f'{len(players)} jogadores', (10, 28), 0, 0.8,
                (0,255,0) if len(players)==4 else (0,0,255), 2)
    out.write(img)
cap.release(); out.release()

import numpy as np
n = np.array(n_por_frame)
print(f'\nNESTE RALLY ({T_INI}s - {T_FIM}s, {len(n)} frames):')
for k in range(0, 7):
    c = (n==k).sum()
    if c: print(f'   {k} jogadores: {c:4d} frames ({100*c/len(n):5.1f}%)')
print(f'\n   >>> 4 jogadores em {100*(n>=4).mean():.1f}% do rally')
print('\nvideo: /content/rally_boxes.mp4')

In [ ]:
# ver no proprio Colab
from IPython.display import HTML
from base64 import b64encode
mp4 = open('/content/rally_boxes.mp4','rb').read()
HTML(f'<video width=900 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

In [ ]:
# descarregar (se preferires ver no Mac)
from google.colab import files
files.download('/content/rally_boxes.mp4')

### O que fazer com o que vires

| O que vês | O que significa |
|---|---|
| **4 caixas quase sempre, IDs estáveis** | ✅ É o que precisamos. Avança para a célula 7. |
| Caixas em **público** (fora do campo) | O polígono dos 4 cantos ficou largo demais para um lado. Não é grave — o `filtrar_espectadores()` do repo limpa isso a partir da calibração do campo. |
| **Falta um jogador** com frequência | Diz-me. Vemos se é oclusão (o ByteTrack devia aguentar) ou o polígono a cortar. |
| **IDs trocam** quando se cruzam | Diz-me. Afina-se o ByteTrack. |

## 7. Descarregar
Guarda o ficheiro na pasta do projeto, em `dados_parada4/`. É este que substitui o
`player_boxes_parada4.pkl` improvisado.

In [ ]:
from google.colab import files
files.download(OUT)